In [1]:
!pip install -q groq python-dotenv
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.1.2
    Uninstalling groq-1.1.2:
      Successfully uninstalled groq-1.1.2


In [2]:
import getpass
import os
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

Enter your Groq API Key: ··········


In [3]:
BASE_MODEL = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Explain errors clearly and provide corrected code if necessary.
Be concise but technically rigorous."""
    },

    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Be empathetic and professional.
Explain refund policies clearly.
Guide users through next steps calmly and politely."""
    },

    "general": {
        "system_prompt": """You are a helpful general customer support assistant.
Respond conversationally and clearly."""
    }
}

In [4]:
def route_prompt(user_input):
    router_prompt = f"""
Classify this text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a classification router."},
            {"role": "user", "content": router_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category

In [5]:
def process_request(user_input):

    # Step 1: Route request
    category = route_prompt(user_input)
    print(f"🔀 Routed to: {category.upper()} expert\n")

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 2: Generate response
    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [6]:
print(process_request("My python script is throwing an IndexError on line 5."))
print(process_request("I was charged twice for my subscription this month."))
print(process_request("What are your business hours?"))

🔀 Routed to: TECHNICAL expert

**IndexError Explanation**

An `IndexError` typically occurs when you try to access an element in a list or other sequence that doesn't exist. This can happen when:

* You're trying to access an index that's out of range (e.g., `my_list[10]` if `my_list` only has 5 elements).
* You're trying to access a non-integer index (e.g., `my_list[3.14]`).

To help you better, could you please provide the following:

1. The code that's throwing the `IndexError`.
2. The line number where the error occurs (you've mentioned it's line 5).
3. The relevant parts of the code surrounding line 5.

This will allow me to give you a more accurate and helpful response.

**Example of how to provide the necessary information**

```python
# Your code here

# Error occurs on line 5
# print(my_list[5])  # This line throws the IndexError
```

Please paste your code, and I'll be happy to assist you in fixing the `IndexError`.
🔀 Routed to: BILLING expert

I'm so sorry to hear that you'v